# Prerequisites

## Load `employee.csv` into DataFrame

In [0]:
df = spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/employee.csv",
    header=True,
    inferSchema=True,
    sep="|",
    quote="'"
)
df.display()

# Selecting columns

Column Object has many built-in functions

In [0]:
# df.select("id","name").display()
from pyspark.sql.functions import col
df.select(col("id"), col("name").alias("full_name")).display()

Select only the name of people who are Male

In [0]:
df.filter(col("gen")=="M").select(col("name")).display()

Selecting the name only who are Male and work in Infosys

In [0]:
df.filter((col("gen") == "M") & (col("company") == "Infosys")).select("name").display()

In [0]:
df.filter(col("gen")=="M").filter(col("company")=="Infosys").select("name").display()

Selecting employees who works for Cisco

Company data has case discrepancies, so using `lower()` method from `pyspark.sql.functions`

In [0]:
from pyspark.sql.functions import lower

# The return type of lower functions is also Column Object
df.filter(lower(col("company")) == "cisco").select("name").display()

## Creating a Column

In [0]:
from pyspark.sql.functions import lit

df.withColumn("is_employed", lit("True")).display()

## Creating a column based on another columns (KPI)

Creating a new column named `exp_category` 

In [0]:
from pyspark.sql.functions import when

df.withColumn(
    "exp_category",
    when(col("exp") >= 10, "Senior")
    .when(col("exp") >= 5, "Mid Level")
    .when(col("exp") >= 0, "Junior")
    .otherwise("Invalid Experience"),
).select("name", "exp", "exp_category").display()

1. If gen="M" then gender = "Male"
2. If gen="F" then gender = "Female"
3. If gen="T" then gender = "Transgender"

## I. Creating a new column named `gender`

In [0]:
df.withColumn(
    "gender",
    when(col("gen") == "M", "Male")
    .when(col("gen") == "F", "Female")
    .when(col("gen") == "T", "Transgender")
    .otherwise("Unknown"),
).display()

## 2. Updating the existing column

## Groupby

In [0]:
df.groupBy("gen").count().display()

GroupBy for experience levels

In [0]:
df2 = df.withColumn(
        "exp_category",
        when(col("exp") >= 10, "Senior")
        .when(col("exp") >= 5, "Mid Level")
        .when(col("exp") >= 0, "Junior")
        .otherwise("Invalid Experience"),
    )

df2.groupBy("exp_category").count().sort("exp_category").display()


1. Senior - Male:10, Female:9
2. Mid Level - Male:5, Female:4

In [0]:
df3 = df.withColumn(
    "exp_category", when((col("exp") >= 10) & (col("gen") == "M"), "Senior")
    .when((col("exp") >= 9) & (col("gen") == "F"), "Senior")
    .when((col("exp") >= 5) & (col("gen") == "M"), "Mid Level")
    .when((col("exp") >= 4) & (col("gen") == "F"), "Mid Level")
    .when(col("exp") >= 0, "Junior")
    .otherwise("Invalid Experience")
)

In [0]:
df3.select("gen", "exp", "exp_category").display()